# 10 — Filesystem-cache startup

Filesystem page-cache state is distinct from compiled-component cache state. No persistent compiled-cache claim is made without artifact identity and hit evidence. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

try:
    batch=resolve_result_batch('e-perf-9', diagnostic_path=os.environ.get('E_PERF_9_DIR'))
except (FileNotFoundError, RuntimeError, ValueError):
    batch=None
artifacts=[] if batch is None else passed_json(batch,'startup.json')
raw_rows=[]
for path,value in artifacts:
    row={
        'condition':path.parent.parent.name,
        'run':path.parent.name,
        'cache_state':value['cache_state'],
        'total_ms':value['total_wall_duration_ns']/1e6,
        'compiled_cache_mode':value['compiled_component_cache']['mode'],
        'compiled_cache_hit':value['compiled_component_cache']['hit'],
    }
    row.update({f'{name}_ms':duration/1e6 for name,duration in value['phases_ns'].items()})
    raw_rows.append(row)
raw=pd.DataFrame(raw_rows)
rows=[]
phase_columns=[
    'component_load_compile_ms','first_process_ms','instantiation_ms',
    'pipeline_setup_ms','process_config_ms',
]
for condition in ['small-cold','small-warm','medium-cold','medium-warm','large-cold','large-warm']:
    values=raw[raw.condition == condition] if not raw.empty else raw
    if values.empty:
        row=pending_record(condition,'no passed startup.json leaf','milliseconds')
        row.update({'condition':condition,'N_runs':0})
        rows.append(row)
    else:
        row={
            'question':condition,
            'condition':condition,
            'status':'READY',
            'N_runs':len(values),
            'cache_state':values['cache_state'].iloc[0],
            'median_total_ms':values['total_ms'].median(),
            'compiled_cache_mode':values['compiled_cache_mode'].iloc[0],
            'compiled_cache_hit':bool(values['compiled_cache_hit'].all()),
            'units':'milliseconds',
            'uncertainty':'descriptive only',
            'thesis_evidence':False,
            'claim_boundary':'filesystem state; no compiled-cache hit claim',
        }
        row.update({f'median_{name}':values[name].median() for name in phase_columns})
        rows.append(row)
df=pd.DataFrame(rows)
print(f"{evidence_label(len(raw), 'milliseconds', False)}; independent runs across {len(df[df.status=='READY'])} conditions")
display(df)
